# Pipeline ANVISA - Download (Stage 1.0)

Notebook de referencia para manutencao da etapa de download e consolidacao bruta da base ANVISA.

Fluxo coberto aqui:
1. Preparar pastas
2. Raspar links da ANVISA
3. Filtrar periodo de coleta
4. Baixar arquivos
5. Limpar arquivos baixados
6. Consolidar CSV bruto final


In [ ]:
import os
import sys
import shutil
import logging
from datetime import datetime
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pipelines").exists():
    raise RuntimeError("Abra este notebook na raiz do projeto (onde existe a pasta 'pipelines').")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

print(f"Projeto: {PROJECT_ROOT}")


In [ ]:
from pipelines.anvisa_base.config_anvisa import (
    ANO_INICIO as CONF_ANO_INICIO,
    MES_INICIO as CONF_MES_INICIO,
    ANO_FIM as CONF_ANO_FIM,
    MES_FIM as CONF_MES_FIM,
    PASTA_DOWNLOADS_BRUTOS,
    PASTA_ARQUIVOS_LIMPOS,
    ARQUIVO_CONSOLIDADO_TEMP,
)

from pipelines.anvisa_base.workflows.baixar import (
    scrape_anvisa_links,
    download_files,
    clean_downloaded_files,
    consolidate_cleaned_files,
)

if not logging.getLogger().handlers:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )


In [ ]:
# Parametros de execucao
FORCE_REFRESH = False

# Defina overrides para testar janelas especificas.
# Se ficar como None, usa o periodo padrao de config_anvisa.py
ANO_INICIO_OVERRIDE = None
MES_INICIO_OVERRIDE = None
ANO_FIM_OVERRIDE = None
MES_FIM_OVERRIDE = None

ano_inicio = ANO_INICIO_OVERRIDE if ANO_INICIO_OVERRIDE is not None else CONF_ANO_INICIO
mes_inicio = MES_INICIO_OVERRIDE if MES_INICIO_OVERRIDE is not None else CONF_MES_INICIO
ano_fim = ANO_FIM_OVERRIDE if ANO_FIM_OVERRIDE is not None else CONF_ANO_FIM
mes_fim = MES_FIM_OVERRIDE if MES_FIM_OVERRIDE is not None else CONF_MES_FIM

data_inicio = datetime(ano_inicio, mes_inicio, 1)
data_fim = datetime(ano_fim, mes_fim, 1)

print(f"Periodo efetivo: {mes_inicio:02d}/{ano_inicio} ate {mes_fim:02d}/{ano_fim}")
print(f"FORCE_REFRESH={FORCE_REFRESH}")
print(f"Downloads brutos: {PASTA_DOWNLOADS_BRUTOS}")
print(f"Arquivos limpos: {PASTA_ARQUIVOS_LIMPOS}")
print(f"Consolidado bruto: {ARQUIVO_CONSOLIDADO_TEMP}")


In [ ]:
# 1) Preparacao de pastas
if FORCE_REFRESH and os.path.exists(PASTA_DOWNLOADS_BRUTOS):
    shutil.rmtree(PASTA_DOWNLOADS_BRUTOS)
    print(f"Pasta removida por force refresh: {PASTA_DOWNLOADS_BRUTOS}")

os.makedirs(PASTA_DOWNLOADS_BRUTOS, exist_ok=True)
os.makedirs(PASTA_ARQUIVOS_LIMPOS, exist_ok=True)

print("Estrutura pronta.")


In [ ]:
# 2) Raspar links da ANVISA
df_links = scrape_anvisa_links()
print(f"Links encontrados: {len(df_links)}")
display(df_links.tail(12))


In [ ]:
# 3) Filtrar o periodo selecionado
df_to_download = df_links[df_links.apply(
    lambda row: data_inicio <= datetime(int(row["ano"]), int(row["mes"]), 1) <= data_fim,
    axis=1
)].copy()

print(f"Links dentro do periodo: {len(df_to_download)}")
if df_to_download.empty:
    print("Nenhum link para baixar no periodo informado.")
else:
    display(df_to_download.tail(12))


In [ ]:
# 4) Download dos arquivos
if df_to_download.empty:
    print("Download ignorado: nenhum link no periodo.")
else:
    download_files(df_to_download)
    print("Download finalizado.")


In [ ]:
# 5) Limpeza dos arquivos e 6) Consolidacao bruta
clean_downloaded_files(PASTA_DOWNLOADS_BRUTOS, PASTA_ARQUIVOS_LIMPOS)
df_consolidado = consolidate_cleaned_files(PASTA_ARQUIVOS_LIMPOS, ARQUIVO_CONSOLIDADO_TEMP)

if df_consolidado is None:
    raise RuntimeError("Consolidacao falhou. Verifique logs das celulas anteriores.")

print(f"Consolidado bruto salvo em: {Path(ARQUIVO_CONSOLIDADO_TEMP).resolve()}")
print(f"Total de linhas: {len(df_consolidado):,}")
display(df_consolidado.head(5))


## Proximo passo

Este notebook cobre apenas a etapa 1.0 (download e consolidacao bruta).
Para seguir no fluxo completo, execute o script principal:

```bash
python 1_download_anvisa.py
```
